# Visualización de Resultados de Entrenamiento

Este notebook agrupa y visualiza las métricas de entrenamiento de los diferentes modelos:
- **CER (Character Error Rate)**: Train y Validation
- **Loss**: Train y Validation
- **Learning Rate**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Directorio con los resultados
data_dir = Path('Data_results')

# Crear carpeta para guardar imágenes
images_dir = Path('images')
images_dir.mkdir(exist_ok=True)
print(f"Carpeta '{images_dir}' creada/verificada")

# Modelos disponibles
models = ['gru_custom', 'gru_mobilenet', 'lstm_custom', 'lstm_mobilenet', 'transformer']

# Función para cargar datos
def load_metric_data(model_name, metric_type):
    """
    Carga datos de una métrica específica para un modelo.
    
    Args:
        model_name: Nombre del modelo (ej: 'gru_custom')
        metric_type: Tipo de métrica (ej: 'CER_train', 'Loss_val', 'Learning_Rate')
    
    Returns:
        DataFrame con los datos
    """
    file_path = data_dir / f'run-{model_name}-tag-{metric_type}.csv'
    if file_path.exists():
        df = pd.read_csv(file_path)
        df['Model'] = model_name
        return df
    return None

# Función para calcular información de entrenamiento
def calculate_training_info(df):
    """
    Calcula el número de épocas y el tiempo total de entrenamiento.
    
    Args:
        df: DataFrame con columnas 'Wall time' y 'Step'
    
    Returns:
        Tupla (número de épocas, tiempo en minutos)
    """
    if df is None or df.empty:
        return 0, 0
    
    num_epochs = df['Step'].max()
    time_seconds = df['Wall time'].max() - df['Wall time'].min()
    time_minutes = time_seconds / 60
    
    return num_epochs, time_minutes

print("Funciones de carga y análisis definidas correctamente")

## 1. Información de Entrenamiento (Épocas y Tiempo)

In [ ]:
# Calcular información de entrenamiento para cada modelo
print("=" * 70)
print("INFORMACIÓN DE ENTRENAMIENTO")
print("=" * 70)

training_info = {}
for model in models:
    # Usar CER_train para obtener información de épocas y tiempo
    df = load_metric_data(model, 'CER_train')
    if df is not None:
        num_epochs, time_minutes = calculate_training_info(df)
        training_info[model] = {'epochs': num_epochs, 'time_minutes': time_minutes}
        print(f"\n{model.upper()}:")
        print(f"  Número de épocas: {num_epochs}")
        print(f"  Tiempo total: {time_minutes:.2f} minutos ({time_minutes/60:.2f} horas)")

print("\n" + "=" * 70)

## 2. CER (Character Error Rate) - Validation

In [ ]:
# Cargar CER de validación para todos los modelos
cer_val_dfs = []
for model in models:
    df = load_metric_data(model, 'CER_val')
    if df is not None:
        cer_val_dfs.append(df)

# Combinar todos los dataframes
cer_val_combined = pd.concat(cer_val_dfs, ignore_index=True)

# Visualizar
plt.figure(figsize=(12, 6))
for model in models:
    model_data = cer_val_combined[cer_val_combined['Model'] == model]
    if not model_data.empty:
        plt.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)

plt.xlabel('Step', fontsize=22)
plt.ylabel('CER (%)', fontsize=22)
plt.title('Character Error Rate (CER) - Validation', fontsize=24, fontweight='bold')
plt.legend(loc='best', fontsize=20)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(images_dir / 'cer_validation.svg', format='svg', bbox_inches='tight')
plt.savefig(images_dir / 'cer_validation.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

# Mostrar estadísticas
print("\nEstadísticas CER Validation:")
print(cer_val_combined.groupby('Model')['Value'].agg(['min', 'max', 'mean', 'std']))

## 3. CER (Character Error Rate) - Training

In [ ]:
# Cargar CER de entrenamiento para todos los modelos
cer_train_dfs = []
for model in models:
    df = load_metric_data(model, 'CER_train')
    if df is not None:
        cer_train_dfs.append(df)

# Combinar todos los dataframes
cer_train_combined = pd.concat(cer_train_dfs, ignore_index=True)

# Visualizar
plt.figure(figsize=(12, 6))
for model in models:
    model_data = cer_train_combined[cer_train_combined['Model'] == model]
    if not model_data.empty:
        plt.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)

plt.xlabel('Step', fontsize=22)
plt.ylabel('CER (%)', fontsize=22)
plt.title('Character Error Rate (CER) - Training', fontsize=24, fontweight='bold')
plt.legend(loc='best', fontsize=20)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(images_dir / 'cer_training.svg', format='svg', bbox_inches='tight')
plt.savefig(images_dir / 'cer_training.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

# Mostrar estadísticas
print("\nEstadísticas CER Training:")
print(cer_train_combined.groupby('Model')['Value'].agg(['min', 'max', 'mean', 'std']))

## 4. Loss - Validation

In [ ]:
# Cargar Loss de validación para todos los modelos
loss_val_dfs = []
for model in models:
    df = load_metric_data(model, 'Loss_val')
    if df is not None:
        loss_val_dfs.append(df)

# Combinar todos los dataframes
loss_val_combined = pd.concat(loss_val_dfs, ignore_index=True)

# Visualizar
plt.figure(figsize=(12, 6))
for model in models:
    model_data = loss_val_combined[loss_val_combined['Model'] == model]
    if not model_data.empty:
        plt.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)

plt.xlabel('Step', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Loss - Validation', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(images_dir / 'loss_validation.svg', format='svg', bbox_inches='tight')
plt.savefig(images_dir / 'loss_validation.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

# Mostrar estadísticas
print("\nEstadísticas Loss Validation:")
print(loss_val_combined.groupby('Model')['Value'].agg(['min', 'max', 'mean', 'std']))

## 5. Loss - Training

In [ ]:
# Cargar Loss de entrenamiento para todos los modelos
loss_train_dfs = []
for model in models:
    df = load_metric_data(model, 'Loss_train')
    if df is not None:
        loss_train_dfs.append(df)

# Combinar todos los dataframes
loss_train_combined = pd.concat(loss_train_dfs, ignore_index=True)

# Visualizar
plt.figure(figsize=(12, 6))
for model in models:
    model_data = loss_train_combined[loss_train_combined['Model'] == model]
    if not model_data.empty:
        plt.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)

plt.xlabel('Step', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Loss - Training', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(images_dir / 'loss_training.svg', format='svg', bbox_inches='tight')
plt.savefig(images_dir / 'loss_training.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

# Mostrar estadísticas
print("\nEstadísticas Loss Training:")
print(loss_train_combined.groupby('Model')['Value'].agg(['min', 'max', 'mean', 'std']))

## 6. Learning Rate

In [ ]:
# Cargar Learning Rate para todos los modelos
lr_dfs = []
for model in models:
    df = load_metric_data(model, 'Learning_Rate')
    if df is not None:
        lr_dfs.append(df)

# Combinar todos los dataframes
lr_combined = pd.concat(lr_dfs, ignore_index=True)

# Visualizar
plt.figure(figsize=(12, 6))
for model in models:
    model_data = lr_combined[lr_combined['Model'] == model]
    if not model_data.empty:
        plt.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)

plt.xlabel('Step', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.title('Learning Rate Evolution', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')  # Escala logarítmica para mejor visualización
plt.tight_layout()
plt.savefig(images_dir / 'learning_rate.svg', format='svg', bbox_inches='tight')
plt.savefig(images_dir / 'learning_rate.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

# Mostrar estadísticas
print("\nEstadísticas Learning Rate:")
print(lr_combined.groupby('Model')['Value'].agg(['min', 'max', 'mean', 'std']))

## 7. Comparación General - Gráficos Combinados

In [ ]:
# Crear una figura con múltiples subplots
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('Comparación de Métricas - Todos los Modelos', fontsize=16, fontweight='bold', y=1.00)

# CER Train
ax = axes[0, 0]
for model in models:
    model_data = cer_train_combined[cer_train_combined['Model'] == model]
    if not model_data.empty:
        ax.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('CER (%)')
ax.set_title('CER - Training')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)

# CER Val
ax = axes[0, 1]
for model in models:
    model_data = cer_val_combined[cer_val_combined['Model'] == model]
    if not model_data.empty:
        ax.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('CER (%)')
ax.set_title('CER - Validation')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)

## Learning Rate
#ax = axes[0, 2]
#for model in models:
#    model_data = lr_combined[lr_combined['Model'] == model]
#    if not model_data.empty:
#        ax.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)
#ax.set_xlabel('Step')
#ax.set_ylabel('Learning Rate')
#ax.set_title('Learning Rate')
#ax.legend(loc='best', fontsize=8)
#ax.grid(True, alpha=0.3)
#ax.set_yscale('log')

# Loss Train
ax = axes[1, 0]
for model in models:
    model_data = loss_train_combined[loss_train_combined['Model'] == model]
    if not model_data.empty:
        ax.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Loss - Training')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)

# Loss Val
ax = axes[1, 1]
for model in models:
    model_data = loss_val_combined[loss_val_combined['Model'] == model]
    if not model_data.empty:
        ax.plot(model_data['Step'], model_data['Value'], label=model, linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Loss - Validation')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)

## Tabla con información de entrenamiento (épocas y tiempo)
#ax = axes[1, 2]
#ax.axis('off')
#
## Crear tabla con épocas y tiempo
#table_data = []
#for model in models:
#    if model in training_info:
#        epochs = training_info[model]['epochs']
#        time_min = training_info[model]['time_minutes']
#        table_data.append([model, f'{epochs}', f'{time_min:.1f} min'])
#
#table = ax.table(cellText=table_data, 
#                colLabels=['Model', 'Épocas', 'Tiempo'],
#                cellLoc='center',
#                loc='center',
#                bbox=[0, 0, 1, 1])
#table.auto_set_font_size(False)
#table.set_fontsize(8)
#table.scale(1, 2)
#ax.set_title('Info Entrenamiento')

plt.tight_layout()
plt.savefig(images_dir / 'comparison_all_metrics.svg', format='svg', bbox_inches='tight')
plt.savefig(images_dir / 'comparison_all_metrics.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Resumen Final

In [ ]:
# Crear resumen con los mejores resultados
print("="*60)
print("RESUMEN DE RESULTADOS - TODOS LOS MODELOS")
print("="*60)

for model in models:
    print(f"\n{model.upper()}:")
    print("-" * 40)
    
    # CER Train
    cer_train_data = cer_train_combined[cer_train_combined['Model'] == model]
    if not cer_train_data.empty:
        print(f"  CER Train - Mínimo: {cer_train_data['Value'].min():.2f}%")
        print(f"  CER Train - Final: {cer_train_data['Value'].iloc[-1]:.2f}%")
    
    # CER Val
    cer_val_data = cer_val_combined[cer_val_combined['Model'] == model]
    if not cer_val_data.empty:
        print(f"  CER Val - Mínimo: {cer_val_data['Value'].min():.2f}%")
        print(f"  CER Val - Final: {cer_val_data['Value'].iloc[-1]:.2f}%")
    
    # Loss Train
    loss_train_data = loss_train_combined[loss_train_combined['Model'] == model]
    if not loss_train_data.empty:
        print(f"  Loss Train - Mínimo: {loss_train_data['Value'].min():.4f}")
        print(f"  Loss Train - Final: {loss_train_data['Value'].iloc[-1]:.4f}")
    
    # Loss Val
    loss_val_data = loss_val_combined[loss_val_combined['Model'] == model]
    if not loss_val_data.empty:
        print(f"  Loss Val - Mínimo: {loss_val_data['Value'].min():.4f}")
        print(f"  Loss Val - Final: {loss_val_data['Value'].iloc[-1]:.4f}")

print("\n" + "="*60)
print("Análisis completado exitosamente!")
print("="*60)

# Listar archivos guardados
print("\n📊 Gráficos guardados:")
print("-" * 40)
print("\nFormato SVG:")
svg_files = sorted(images_dir.glob('*.svg'))
for svg_file in svg_files:
    print(f"  ✓ {svg_file.name}")

print("\nFormato PNG:")
png_files = sorted(images_dir.glob('*.png'))
for png_file in png_files:
    print(f"  ✓ {png_file.name}")

print(f"\nTotal: {len(svg_files)} archivos SVG + {len(png_files)} archivos PNG")
print(f"Guardados en: '{images_dir}/'")
print("="*60)